## Parte 1 — Softmax con temperatura

La softmax con temperatura se define como p_i = exp(z_i/T) / Σ_j exp(z_j/T). Para hacerla numéricamente estable se resta el máximo de los logits antes de exponenciar: p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T). Como z_max = 2.0, todos los exponentes resultan ≤ 0 y no hay desbordamiento; el factor común exp(z_max/T) se cancela entre numerador y denominador, así que el resultado es idéntico al de la fórmula directa.

Con los logits z = [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] de los tokens t0…t5, y la entropía en bits H = −Σ_i p_i·log2(p_i), se obtiene la siguiente tabla (cuatro decimales). Cada fila suma 1.0000:

| Temperatura | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|---|---|---|---|---|---|---|---|
| T=0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| T=1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| T=2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

El efecto de T es evidente: al bajar de T=2 a T=0.5 la masa se concentra en t0 (0.3583 → 0.8231) y la entropía cae de 2.2888 a 0.9147 bits; al subir T la distribución se aplana (t5 pasa de 0.0000 a 0.0294) y la entropía crece, con 1.7998 bits en T=1.

In [1]:
# T1: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==================================================================
# SUBTAREA T1 — Softmax con temperatura (NumPy, estable) + entropía en bits
# ==================================================================

LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TOKENS = [f"token_{i+1}" for i in range(LOGITS.size)]
TEMPERATURAS = [0.5, 1.0, 2.0]


def softmax_con_temperatura(z, T):
    """p_i = exp(z_i/T) / sum_j exp(z_j/T).

    Numéricamente estable: se resta el máximo de los logits escalados
    antes de exponenciar (no cambia la distribución, evita overflow).
    """
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura T debe ser positiva.")
    z_escalado = z / T
    z_escalado = z_escalado - np.max(z_escalado)  # estabilidad numérica
    exp_z = np.exp(z_escalado)
    return exp_z / np.sum(exp_z)


def entropia_en_bits(p):
    """H(p) = -sum p_i log2(p_i), ignorando entradas nulas (0*log0 = 0)."""
    p = np.asarray(p, dtype=float)
    mascara = p > 0
    return float(-np.sum(p[mascara] * np.log2(p[mascara])))


# ---------------- Distribuciones y entropías ----------------
distribuciones = {T: softmax_con_temperatura(LOGITS, T) for T in TEMPERATURAS}
entropias = {T: entropia_en_bits(distribuciones[T]) for T in TEMPERATURAS}
sumas = {T: float(distribuciones[T].sum()) for T in TEMPERATURAS}

# Verificación: cada distribución suma 1
for T in TEMPERATURAS:
    assert np.isclose(sumas[T], 1.0, atol=1e-12), f"T={T} no suma 1"

# ---------------- Tabla (4 decimales) ----------------
# Una fila por temperatura: 6 probabilidades + entropía en bits.
tabla = pd.DataFrame(
    data=[np.concatenate([distribuciones[T], [entropias[T]]]) for T in TEMPERATURAS],
    index=[f"T={T}" for T in TEMPERATURAS],
    columns=TOKENS + ["Entropia_bits"],
)
tabla_celdas = [[f"{v:.4f}" for v in fila] for fila in tabla.to_numpy()]

encabezado = "| Temperatura | " + " | ".join(TOKENS) + " | Entropía (bits) |"
separador = "|" + "---|" * (len(TOKENS) + 2)
filas_md = [
    f"| {idx} | " + " | ".join(celdas) + " |"
    for idx, celdas in zip(tabla.index, tabla_celdas)
]
tabla_markdown = "\n".join([encabezado, separador] + filas_md)

# ---------------- Figura: efecto de la temperatura ----------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(TOKENS))
ancho = 0.25
for k, T in enumerate(TEMPERATURAS):
    ax.bar(x + (k - 1) * ancho, distribuciones[T], width=ancho, label=f"T={T}")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura (logits 2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
fig.tight_layout()
fig.savefig("softmax_temperaturas.png", dpi=120)
plt.close(fig)

# ---------------- resultados.json (contrato) ----------------
resultados = {
    "subtarea": "T1_softmax_con_temperatura",
    "logits": LOGITS.tolist(),
    "tokens": TOKENS,
    "temperaturas": TEMPERATURAS,
    "distribuciones": {f"T={T}": distribuciones[T].tolist() for T in TEMPERATURAS},
    "entropias_bits": {f"T={T}": entropias[T] for T in TEMPERATURAS},
    "suma_de_cada_distribucion": {f"T={T}": sumas[T] for T in TEMPERATURAS},
    "tabla_cuatro_decimales": {
        "columnas": TOKENS + ["Entropia_bits"],
        "filas": list(tabla.index),
        "valores": tabla_celdas,
    },
    "tabla_markdown": tabla_markdown,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------------- Resumen ----------------
print("T1 — Softmax con temperatura (distribución y entropía en bits)")
print(tabla_markdown)
print("\nSuma de cada fila de probabilidades:",
      {f"T={T}": f"{sumas[T]:.12f}" for T in TEMPERATURAS})
print("Figura guardada: softmax_temperaturas.png")
print("Resultados escritos en resultados.json")


T1 — Softmax con temperatura (distribución y entropía en bits)
| Temperatura | token_1 | token_2 | token_3 | token_4 | token_5 | token_6 | Entropía (bits) |
|---|---|---|---|---|---|---|---|
| T=0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| T=1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| T=2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Suma de cada fila de probabilidades: {'T=0.5': '1.000000000000', 'T=1.0': '1.000000000000', 'T=2.0': '1.000000000000'}
Figura guardada: softmax_temperaturas.png
Resultados escritos en resultados.json


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1 de la Parte 1 se aplica top-p con p = 0.9. Los tokens se ordenan por probabilidad descendente; como los logits ya son decrecientes, el orden es t0, t1, t2, t3, t4, t5. La probabilidad acumulada es 0.551623, 0.754553, 0.877637, 0.968820, 0.996283, 1.000000. Con tres tokens la acumulada es 0.877637 < 0.9, por lo que el conjunto mínimo que alcanza el umbral requiere añadir t3: la acumulada del núcleo queda en 0.968820 ≥ 0.9. Sobreviven t0, t1, t2 y t3; se descartan t4 y t5, cuya masa conjunta es 0.0312. Renormalizando (dividiendo entre 0.968820), la distribución resultante suma 1.0:

| Token | p_original | p_acumulada | p_renormalizada |
|---|---|---|---|
| t0 | 0.551623 | 0.551623 | 0.569376 |
| t1 | 0.202931 | 0.754553 | 0.209462 |
| t2 | 0.123084 | 0.877637 | 0.127045 |
| t3 | 0.091183 | 0.968820 | 0.094117 |

El recorte apenas altera las probabilidades relativas dentro del núcleo (p. ej., t0 pasa de 0.551623 a 0.569376) porque la masa descartada es pequeña.

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# T2 — Muestreo de núcleo (top-p, p = 0.9) sobre la
# distribución de T = 1 obtenida en la Parte 1 (T1).
# ============================================================

# ---------- 1) Cargar la distribución de T=1 de la Parte 1 ----------
ruta_t1 = Path("entrada/T1/resultados.json")
if ruta_t1.exists():
    with open(ruta_t1, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    tokens = list(t1["tokens"])
    logits = np.asarray(t1["logits"], dtype=float)
    probs = np.asarray(t1["distribuciones"]["T=1.0"], dtype=float)
else:
    # Respaldo: recomputar la softmax numéricamente estable con T=1
    tokens = ["token_1", "token_2", "token_3", "token_4", "token_5", "token_6"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    e = np.exp(z)
    probs = e / e.sum()

# Verificación de consistencia: la distribución cargada debe coincidir
# con la softmax estable (T=1) recalculada a partir de los logits.
z = logits - logits.max()
probs_ref = np.exp(z) / np.exp(z).sum()
max_diff = float(np.max(np.abs(probs - probs_ref)))

# ---------- 2) Top-p (núcleo) con p = 0.9 ----------
p = 0.9
orden = np.argsort(-probs, kind="stable")            # orden descendente de probabilidad
probs_ord = probs[orden]
tokens_ord = [tokens[i] for i in orden]
acum = np.cumsum(probs_ord)

# Conjunto MÍNIMO cuya probabilidad acumulada alcanza p:
# primer índice j (0-based) con acum[j] >= p  ->  k = j + 1 tokens sobreviven
j = int(np.searchsorted(acum, p, side="left"))
k = j + 1
idx_sup = orden[:k]
idx_desc = orden[k:]

tokens_sup = [tokens[i] for i in idx_sup]
probs_sup = probs[idx_sup]
acum_sup_lista = np.cumsum(probs_sup)                # acumulada dentro del núcleo
acum_sup = float(acum[j])                            # prob. acumulada del núcleo (>= 0.9)
renorm = probs_sup / probs_sup.sum()                 # distribución renormalizada

tokens_desc = [tokens[i] for i in idx_desc]
probs_desc = probs[idx_desc]

minimal = bool(j == 0 or acum[j - 1] < p)            # el token previo NO alcanzaba p

# ---------- 3) Figura ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

x = np.arange(len(tokens))
colores = ["#2a9d8f" if tk in tokens_sup else "#e76f51" for tk in tokens_ord]
axes[0].bar(x, probs_ord, color=colores, label="prob. token")
axes[0].plot(x, acum, "o--", color="#264653", label="prob. acumulada")
axes[0].axhline(p, color="gray", ls=":", label="p = 0.9")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens_ord, rotation=30)
axes[0].set_ylabel("probabilidad")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Distribución T=1 ordenada (verde = núcleo top-p)")
axes[0].legend(fontsize=8)

x2 = np.arange(k)
axes[1].bar(x2, renorm, color="#2a9d8f")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(tokens_sup, rotation=30)
axes[1].set_ylabel("probabilidad renormalizada")
axes[1].set_ylim(0, 1.0)
axes[1].set_title(f"Distribución renormalizada (suma = {renorm.sum():.6f})")
for xi, vi in zip(x2, renorm):
    axes[1].text(xi, vi + 0.02, f"{vi:.4f}", ha="center", fontsize=8)

fig.tight_layout()
fig.savefig("T2_top_p_renormalizada.png", dpi=120)
plt.close(fig)

# ---------- 4) resultados.json ----------
tabla_md = "| Token | p_original | p_acumulada | p_renormalizada |\n|---|---|---|---|\n"
for tk, po, pa, pr in zip(tokens_sup, probs_sup, acum_sup_lista, renorm):
    tabla_md += f"| {tk} | {po:.6f} | {pa:.6f} | {pr:.6f} |\n"

resultados = {
    "subtarea": "T2_muestreo_nucleo_top_p",
    "p": p,
    "distribucion_base": "T=1.0 (Parte 1)",
    "tokens": tokens,
    "distribucion_original_T1": {tk: float(v) for tk, v in zip(tokens, probs)},
    "verificacion_softmax_T1_max_diff": max_diff,
    "tokens_ordenados_descendente": tokens_ord,
    "probs_ordenadas_descendente": [float(v) for v in probs_ord],
    "prob_acumulada_ordenada": [float(v) for v in acum],
    "num_tokens_supervivientes": k,
    "tokens_supervivientes": tokens_sup,
    "probs_supervivientes_originales": [float(v) for v in probs_sup],
    "prob_acumulada_por_token_superviviente": [float(v) for v in acum_sup_lista],
    "prob_acumulada_nucleo": acum_sup,
    "cumple_umbral_p": bool(acum_sup >= p),
    "conjunto_minimo": minimal,
    "tokens_descartados": tokens_desc,
    "probs_descartados": [float(v) for v in probs_desc],
    "masa_descartada": float(probs_desc.sum()),
    "distribucion_renormalizada": {tk: float(v) for tk, v in zip(tokens_sup, renorm)},
    "suma_distribucion_renormalizada": float(renorm.sum()),
    "tabla_markdown": tabla_md,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------- 5) Resumen ----------
print("T2 — Muestreo de núcleo top-p (p=0.9) sobre la distribución T=1 de la Parte 1")
print(f"Tokens ordenados (desc): {tokens_ord}")
print(f"Prob. acumulada: {[round(v, 6) for v in acum]}")
print(f"Núcleo (conjunto mínimo que alcanza 0.9): {tokens_sup}  | acumulada = {acum_sup:.6f} (>= 0.9: {acum_sup >= p}, mínimo: {minimal})")
print(f"Descartados: {tokens_desc} | masa descartada = {probs_desc.sum():.6f}")
print("Distribución renormalizada:")
for tk, v in zip(tokens_sup, renorm):
    print(f"  {tk}: {v:.6f}")
print(f"Suma de la distribución renormalizada = {renorm.sum():.12f}")
print("Figura guardada: T2_top_p_renormalizada.png")
print("Resultados escritos en resultados.json")


T2 — Muestreo de núcleo top-p (p=0.9) sobre la distribución T=1 de la Parte 1
Tokens ordenados (desc): ['token_1', 'token_2', 'token_3', 'token_4', 'token_5', 'token_6']
Prob. acumulada: [np.float64(0.551623), np.float64(0.754553), np.float64(0.877637), np.float64(0.96882), np.float64(0.996283), np.float64(1.0)]
Núcleo (conjunto mínimo que alcanza 0.9): ['token_1', 'token_2', 'token_3', 'token_4']  | acumulada = 0.968820 (>= 0.9: True, mínimo: True)
Descartados: ['token_5', 'token_6'] | masa descartada = 0.031180
Distribución renormalizada:
  token_1: 0.569376
  token_2: 0.209462
  token_3: 0.127045
  token_4: 0.094117
Suma de la distribución renormalizada = 1.000000000000
Figura guardada: T2_top_p_renormalizada.png
Resultados escritos en resultados.json


## Parte 3 — Comprobación empírica

Se toman 10,000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados son t0: 5682, t1: 2093, t2: 1296, t3: 929, lo que da frecuencias 0.5682, 0.2093, 0.1296 y 0.0929 (suma 1.0), frente a las teóricas 0.569376, 0.209462, 0.127045 y 0.094117. La celda siguiente genera el gráfico de barras que compara ambas: las diferencias son pequeñas y atribuibles al ruido de muestreo (la mayor, en t2: 0.1296 observado frente a 0.1270 teórico).

La divergencia KL de la empírica respecto de la teórica es KL(empírica ‖ teórica) = 5.0069e-05 bits, un valor prácticamente nulo que confirma que el generador reproduce la distribución objetivo con la semilla fijada.

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# T3: Comprobación empírica de la distribución renormalizada (top-p=0.9)
# 1) Muestrear 10,000 veces con numpy.random.default_rng(0)
# 2) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# 3) Divergencia KL en bits de la empírica respecto de la teórica
# ------------------------------------------------------------------

# --- 1. Cargar la distribución renormalizada de la subtarea T2 ---
ruta_t2 = Path("entrada/T2/resultados.json")
with open(ruta_t2, "r", encoding="utf-8") as f:
    res_t2 = json.load(f)

tokens = list(res_t2["tokens_supervivientes"])
probs_teoricas = np.array(
    [res_t2["distribucion_renormalizada"][t] for t in tokens], dtype=float
)
probs_teoricas = probs_teoricas / probs_teoricas.sum()  # seguridad numérica

# --- 2. Muestreo de 10,000 muestras con semilla 0 ---
N = 10000
rng = np.random.default_rng(0)
indices = np.arange(len(tokens))
muestras = rng.choice(indices, size=N, p=probs_teoricas)

conteos = np.bincount(muestras, minlength=len(tokens))
frecuencias_obs = conteos / N

# --- 3. Divergencia KL en bits: D_KL(empírica || teórica) ---
# KL(q||p) = sum_i q_i * log2(q_i / p_i), con la convención 0*log(0/p)=0
mask = frecuencias_obs > 0
kl_bits = float(
    np.sum(
        frecuencias_obs[mask]
        * np.log2(frecuencias_obs[mask] / probs_teoricas[mask])
    )
)

# --- 4. Gráfico de barras: observado vs teórico ---
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(
    x - ancho / 2, frecuencias_obs, width=ancho,
    label="Frecuencia observada (empírica)", color="#4C72B0", edgecolor="black",
)
b2 = ax.bar(
    x + ancho / 2, probs_teoricas, width=ancho,
    label="Probabilidad teórica (renormalizada top-p=0.9)",
    color="#DD8452", edgecolor="black",
)
for rect in list(b1) + list(b2):
    h = rect.get_height()
    ax.annotate(
        f"{h:.4f}",
        xy=(rect.get_x() + rect.get_width() / 2, h),
        xytext=(0, 3), textcoords="offset points",
        ha="center", va="bottom", fontsize=8,
    )
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad / Frecuencia relativa")
ax.set_ylim(0, max(frecuencias_obs.max(), probs_teoricas.max()) * 1.18)
ax.set_title(
    f"Muestreo de la distribución renormalizada (núcleo top-p = 0.9)\n"
    f"N = {N} muestras, semilla = 0  |  KL(empírica || teórica) = {kl_bits:.6f} bits"
)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("barras_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# --- 5. Escribir resultados.json (contrato de la subtarea) ---
resultados = {
    "subtarea": "T3_comprobacion_empirica_kl",
    "distribucion_muestreada": "renormalizada top-p=0.9 (Parte 2)",
    "num_muestras": N,
    "semilla": 0,
    "generador": "numpy.random.default_rng(0)",
    "tokens": tokens,
    "probabilidades_teoricas": {
        t: float(p) for t, p in zip(tokens, probs_teoricas)
    },
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {
        t: float(fr) for t, fr in zip(tokens, frecuencias_obs)
    },
    "suma_frecuencias_observadas": float(frecuencias_obs.sum()),
    "divergencia_KL_bits": kl_bits,
    "figura": "barras_frecuencias_vs_teoricas.png",
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# --- 6. Resumen ---
print("=== T3: Comprobación empírica de la distribución renormalizada ===")
print(f"Muestras: {N} | Generador: numpy.random.default_rng(0)")
print(f"{'Token':<10}{'Conteo':>8}{'Freq. obs.':>14}{'P teórica':>14}")
for t, c, fo, pt in zip(tokens, conteos, frecuencias_obs, probs_teoricas):
    print(f"{t:<10}{c:>8d}{fo:>14.6f}{pt:>14.6f}")
print(f"\nDivergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: barras_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


=== T3: Comprobación empírica de la distribución renormalizada ===
Muestras: 10000 | Generador: numpy.random.default_rng(0)
Token       Conteo    Freq. obs.     P teórica
token_1       5682      0.568200      0.569376
token_2       2093      0.209300      0.209462
token_3       1296      0.129600      0.127045
token_4        929      0.092900      0.094117

Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: barras_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0**, el término exp(z_i/T) queda dominado por el logit mayor: la distribución tiende a una masa puntual sobre t0 (el token de mayor logit, 2.0) y la entropía tiende a 0. Las entropías medidas muestran esa trayectoria: 2.2888 bits en T=2, 1.7998 bits en T=1 y 0.9147 bits en T=0.5, con p(t0) creciendo 0.3583 → 0.5516 → 0.8231; extrapolando la tendencia, en el límite p(t0) → 1 y H → 0.

**Cuando T → ∞**, los cocientes z_i/T tienden a 0, todos los exponenciales tienden a 1 y la distribución tiende a la uniforme (1/6 por token). La evidencia es la dirección opuesta: al subir T la probabilidad de t0 cae (0.8231 en T=0.5, 0.5516 en T=1, 0.3583 en T=2), la de los tokens raros sube (t5: 0.0000 → 0.0037 → 0.0294) y la entropía crece monótonamente (0.9147 → 1.7998 → 2.2888 bits), acercándose al máximo posible de una distribución uniforme sobre seis tokens.

**Por qué T → 0 equivale a la decodificación voraz:** para cualquier T > 0, la softmax es monótona en los logits (p_i/p_j = exp((z_i−z_j)/T) > 1 si y solo si z_i > z_j), de modo que el token más probable de la distribución es siempre el argmax de los logits, t0. La decodificación voraz selecciona exactamente ese argmax. Cuando T → 0, la probabilidad de t0 tiende a 1, de forma que muestrear de la distribución devuelve con probabilidad 1 el mismo token que elegiría greedy: el muestreo con temperatura se vuelve determinista y coincide con la decodificación voraz en ese límite, tal como indican la concentración de la masa en t0 y la caída de la entropía hacia 0 bits.